In [25]:
import pandas as pd

df = pd.read_csv('properties.csv') 


In [26]:
df.head()          # Preview the first 5 rows

,type,title,location,bedroom,bathroom,size_sqm,price
0,Duplex,Prime Location Duplex Fully Finished With A\C,"Park View, North Investors Area, New Cairo Cit...",4,4,345,"6,850,000"
1,Villa,Town house resale at Mivida Emaar with best price,"Mivida, 5th Settlement Compounds, The 5th Sett...",3,3,285,"10,000,000"
2,Apartment,Lake View Residence - Apartment | Prime Location,"Lake View Residence, 5th Settlement Compounds,...",3,3,210,"5,700,000"
3,Townhouse,Best Penthouse for sale in villette ( sky conds ),"La Vista City, New Capital Compounds, New Capi...",4,4,230,"7,510,000"
4,Penthouse,2nd Floor | Fully Finished | Lowest Price |Par...,"Villette, 5th Settlement Compounds, The 5th Se...",5,6,284,"8,511,300"


In [27]:
df.info()          # Data types, non-null counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11418 entries, 0 to 11417
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   type      11418 non-null  object
 1   title     11418 non-null  object
 2   location  11418 non-null  object
 3   bedroom   11418 non-null  object
 4   bathroom  11418 non-null  int64 
 5   size_sqm  11418 non-null  object
 6   price     11418 non-null  object
dtypes: int64(1), object(6)
memory usage: 624.5+ KB


In [28]:
df.describe()      # Summary stats for numerical columns

,bathroom
count,11418.000000
mean,3.328604
std,1.312016
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,8.000000


In [29]:
df.columns         # Check column names

Index(['type', 'title', 'location', 'bedroom', 'bathroom', 'size_sqm',
       'price'],
      dtype='object')

In [30]:
null_array = round(df.isnull().sum()/len(df)*100,2).sort_values().to_frame().rename(columns= {0:"Train % of Missing Values"})
print(null_array, end='\n')

          Train % of Missing Values
type                            0.0
title                           0.0
location                        0.0
bedroom                         0.0
bathroom                        0.0
size_sqm                        0.0
price                           0.0


In [31]:
for column in df.columns:
    print(f"Value counts for column '{column}':", end='/n')
    print(df[column].value_counts())
    print("\n----------------------\n")

Value counts for column 'type':/ntype
Apartment          5848
Villa              2845
Townhouse           858
Twin House          601
Duplex              568
Penthouse           448
iVilla              199
Hotel Apartment      34
Chalet               14
Compound              3
Name: count, dtype: int64

----------------------

Value counts for column 'title':/ntitle
Villa for sale In Lake View L:600 BUE:550 price18M                 14
STANDALONE in Palm Hills Katameya EX.PK2 For Sale                  13
2 bedrooms|2 bath|Terrace|with disc                                13
3 bedrooms|7 years install|Ready to move|Disc 12%                  12
Move Now to Standalone Villa with 5% DP over 8 years Sodic East     9
                                                                   ..
Pay Only 5% DP | Over 9 Years by Tatweer Misr                       1
Apartment 88 m fully finished without over loading                  1
Penthouse In Katameya Plaza Sodic Fully Finished                    1
P

In [32]:
df_wrangled = df.copy()

# Lowercase all string based columns 
for column in ['title','location','type']:
    df_wrangled[column] = df_wrangled[column].apply(lambda m: m.lower())

In [33]:
# remove columns with price value "ask"
df_wrangled = df_wrangled[df_wrangled['price'] != 'Ask']

In [34]:
# trasform bedroom value "Studio" => 1 
df_wrangled['bedroom'] = df_wrangled['bedroom'].replace('Studio', 1)

In [35]:
# remove delimiter "," from "price, size, square km columns "
df_wrangled.price=df_wrangled.price.apply(lambda m:m.replace(',',''))
df_wrangled.size_sqm =df_wrangled.size_sqm.apply(lambda m:m.replace(',',''))

In [36]:
# Convert [price, size_sqm & bedroom] to numeric
for column in ['price','size_sqm','bedroom']:
    df_wrangled[column] = df_wrangled[column].apply(pd.to_numeric, errors='coerce').round(0)


In [40]:
df_wrangled['bedroom'] = df_wrangled['bedroom'].fillna(df_wrangled['bedroom'].median())

In [41]:
# Validate casting
df_wrangled.info()


<class 'pandas.core.frame.DataFrame'>
Index: 11006 entries, 0 to 11417
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      11006 non-null  object 
 1   title     11006 non-null  object 
 2   location  11006 non-null  object 
 3   bedroom   11006 non-null  float64
 4   bathroom  11006 non-null  int64  
 5   size_sqm  11006 non-null  int64  
 6   price     11006 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 687.9+ KB


In [42]:
# Validate
for column in ['price','bedroom']:
    print(f"Value counts for column '{column}':", end='/n')
    print(df_wrangled[column].value_counts())
    print("\n----------------------\n")

Value counts for column 'price':/nprice
2500000    174
3000000    168
3500000    154
4000000    131
2000000    128
          ... 
3999138      1
6180000      1
2555559      1
2270750      1
4554066      1
Name: count, Length: 2496, dtype: int64

----------------------

Value counts for column 'bedroom':/nbedroom
3.0    5444
4.0    2372
2.0    1426
5.0     961
6.0     360
8.0     177
1.0     173
7.0      93
Name: count, dtype: int64

----------------------



In [44]:
df_wrangled.to_csv('properties_cleaned_data.csv', index=False)
